# TDC-KV Colab Validation and Pilot
Run these cells in order on a GPU runtime.

In [ ]:
# 1. Check the runtime
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('PyTorch CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# 2. Clone or update branch-h
REPO_URL = 'https://github.com/JayGor-13/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring.git'
REPO_BRANCH = 'branch-h'
PROJECT_DIR = Path('/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring')

if (PROJECT_DIR / '.git').exists():
    os.chdir(PROJECT_DIR)
    subprocess.run(['git', 'fetch', 'origin'], check=True)
    subprocess.run(['git', 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
        REPO_URL, str(PROJECT_DIR)
    ], check=True)
    os.chdir(PROJECT_DIR)

print('Project:', Path.cwd())
subprocess.run(['git', 'branch', '--show-current'], check=True)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

In [ ]:
# 3. Install text-model dependencies without replacing Colab PyTorch
packages = [
    'transformers>=4.43,<6',
    'datasets>=5.0.1',
    'accelerate>=1.14.0',
    'numpy>=2.5.1',
    'scipy>=1.18.0',
    'matplotlib>=3.11.1',
    'pytest>=9.1.1',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y',
    'torchvision', 'torchaudio'
], check=True)
os.environ['PYTHONPATH'] = str(Path.cwd()) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Dependency setup complete.')

## Kernel checkpoint
If Transformers, TorchVision, or TorchAudio was imported before Cell 3, restart the kernel now, rerun Cells 1-2, and continue at Cell 4.

In [ ]:
# 4. Verify the clean model environment
import transformers
import datasets

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('TorchVision installed:', importlib.util.find_spec('torchvision') is not None)
print('TorchAudio installed:', importlib.util.find_spec('torchaudio') is not None)

from transformers import (
    GPT2Config, GPT2LMHeadModel,
    LlamaConfig, LlamaForCausalLM,
    Qwen2Config, Qwen2ForCausalLM,
)
print('GPT-2, Llama, and Qwen2 imports: OK')

In [ ]:
# 5. Run cache compatibility and end-to-end tests
result = subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_hf_cache_adapter.py',
    'tests/test_hf_cache_e2e.py',
    '-q', '-x', '--tb=long'
], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('Cache compatibility gate failed.')

In [ ]:
# 6. Run the complete test suite
result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('Full test suite failed.')

In [ ]:
# 7. Run the 10-sample GSM8K pilot
OUTPUT = Path('outputs/qwen05b_gsm8k_pilot.json')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, 'scripts/run_hf_grid.py',
    '--models', 'Qwen/Qwen2.5-0.5B-Instruct',
    '--datasets', (
        'name=gsm8k,source=openai/gsm8k,config=main,split=test,'
        'prompt_field=question,answer_field=answer'
    ),
    '--methods', 'fullkv,tdc_kv',
    '--budget-ratios', '0.75,0.5,0.25',
    '--thetas', '0.3',
    '--recent-windows', '16',
    '--alphas', '0.6',
    '--prefill-block-size', '128',
    '--max-samples', '10',
    '--max-length', '1024',
    '--max-new-tokens', '128',
    '--device', 'auto',
    '--dtype', 'auto',
    '--allow-level2-fallback',
    '--output', str(OUTPUT),
]
result = subprocess.run(command, text=True)
if result.returncode != 0:
    raise RuntimeError('GSM8K pilot failed; inspect the output above.')

In [ ]:
# 8. Inspect and validate grouped results
with OUTPUT.open('r', encoding='utf-8') as handle:
    data = json.load(handle)
print(json.dumps(data['summary'], indent=2))

assert data['summary']['failed_runs'] == 0
tdc_groups = [g for g in data['grouped_results'] if g['method'] == 'tdc_kv']
observed_ratios = {
    float(g['budget']['value'])
    for g in tdc_groups
    if g['budget']['type'] == 'ratio'
}
assert observed_ratios == {0.75, 0.5, 0.25}
for group in tdc_groups:
    violations = group.get('decode_cache_summary', {}).get('budget_violations', {}).get('sum', 0)
    assert violations == 0, (group['budget'], violations)
empty_predictions = [
    run for run in data['runs']
    if run.get('status') == 'ok'
    and run.get('method') == 'tdc_kv'
    and not str(run.get('evicted_prediction', '')).strip()
]
assert not empty_predictions
print('Pilot validation passed:', sorted(observed_ratios, reverse=True))